In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import threading
import time
import os
import platform

# --- Configuration ---
WINDOW_NAME = "AI Fitness Trainer"
COLOR_BG = (20, 20, 20)      
COLOR_TEXT = (255, 255, 255) 
COLOR_ACCENT = (255, 190, 0) 
COLOR_GOOD = (0, 255, 0)     
COLOR_BAD = (0, 0, 255)      
COLOR_WARN = (0, 165, 255)   

# --- SYSTEM NATIVE VOICE SETUP (NO PYTTSX3) ---
def system_speak(text):
    """Forces the OS to speak using terminal commands"""
    system_name = platform.system()
    if system_name == "Windows":
        # Uses PowerShell's built-in speech synthesizer
        # The -Async flag makes it not freeze your video
        command = f'powershell -c "Add-Type –AssemblyName System.Speech; (New-Object System.Speech.Synthesis.SpeechSynthesizer).Speak(\'{text}\');"'
        threading.Thread(target=os.system, args=(command,), daemon=True).start()
    elif system_name == "Darwin": # Mac
        os.system(f'say "{text}" &')
    else: # Linux
        os.system(f'espeak "{text}" &')

last_speech_time = 0
speech_cooldown = 2.5

def speak(text):
    global last_speech_time
    current_time = time.time()
    if current_time - last_speech_time > speech_cooldown:
        last_speech_time = current_time
        print(f"🔊 AI Says: {text}") # Debug print
        system_speak(text)

# --- STARTUP CHECK ---
print("System Starting...")
speak("System Ready")

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180.0 else angle

def draw_info_panel(image, reps, stage, feedback, angle_feedback):
    h, w, _ = image.shape
    overlay = image.copy()
    cv2.rectangle(overlay, (0, 0), (250, h), (0, 0, 0), -1)
    alpha = 0.7
    cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)
    
    cv2.putText(image, "REPS", (20, 50), cv2.FONT_HERSHEY_DUPLEX, 0.8, (180, 180, 180), 1)
    cv2.putText(image, str(reps), (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 2, COLOR_TEXT, 3)
    cv2.line(image, (20, 140), (230, 140), (100, 100, 100), 1)
    
    cv2.putText(image, "STAGE", (20, 180), cv2.FONT_HERSHEY_DUPLEX, 0.6, (180, 180, 180), 1)
    stage_text = stage.upper() if stage else "-"
    stage_color = COLOR_GOOD if stage == "up" else COLOR_ACCENT
    cv2.putText(image, stage_text, (20, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, stage_color, 2)
    
    cv2.line(image, (20, 250), (230, 250), (100, 100, 100), 1)
    
    status_color = COLOR_BAD if "Bad" in feedback else COLOR_TEXT
    cv2.putText(image, feedback, (20, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.65, status_color, 2)
    
    if angle_feedback:
        if len(angle_feedback) > 18:
            cv2.putText(image, angle_feedback[:18], (20, 340), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_WARN, 1)
            cv2.putText(image, angle_feedback[18:], (20, 370), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_WARN, 1)
        else:
            cv2.putText(image, angle_feedback, (20, 340), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_WARN, 1)

def draw_progress_bar(image, angle):
    h, w, _ = image.shape
    val = np.clip(angle, 30, 160)
    progress = 1 - ((val - 30) / (160 - 30))
    bar_x, bar_y, bar_w, bar_h = 210, 350, 20, 200
    
    cv2.rectangle(image, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
    fill_h = int(bar_h * progress)
    color = COLOR_ACCENT if progress < 0.8 else COLOR_GOOD
    cv2.rectangle(image, (bar_x, bar_y + bar_h - fill_h), (bar_x + bar_w, bar_y + bar_h), color, -1)
    cv2.putText(image, f"{int(progress*100)}%", (bar_x - 5, bar_y + bar_h + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

cap = cv2.VideoCapture(0)
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 1280, 720)

counter, stage, ready = 0, None, False
last_elbow_x = None
bad_form = False
angle_feedback = "" 
prev_angle = 0  
current_angle = 0

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1) 
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        h, w, _ = image.shape

        feedback_text = "Get Ready"

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            elbow = landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]
            wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
            hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]

            visible = all(l.visibility > 0.6 for l in [shoulder, elbow, wrist, hip])
            within_frame = all(0 < l.x < 1 and 0 < l.y < 1 for l in [shoulder, elbow, wrist, hip])
            dist = np.linalg.norm(np.array([shoulder.x*w, shoulder.y*h]) - np.array([hip.x*w, hip.y*h]))
            too_close, too_far = dist < 90, dist > 260 

            if ready:
                coords = np.array([[shoulder.x*w, shoulder.y*h], [elbow.x*w, elbow.y*h],
                                   [wrist.x*w, wrist.y*h], [hip.x*w, hip.y*h]], dtype=int)
                x_min, y_min = np.min(coords[:, 0]), np.min(coords[:, 1])
                x_max, y_max = np.max(coords[:, 0]), np.max(coords[:, 1])
                
                box_color = COLOR_BAD if bad_form else (COLOR_GOOD if stage == "up" else COLOR_ACCENT)
                pad = 20
                cv2.rectangle(image, (x_min-pad, y_min-pad), (x_max+pad, y_max+pad), box_color, 1)
                len_corner = 20
                cv2.line(image, (x_min-pad, y_min-pad), (x_min-pad+len_corner, y_min-pad), box_color, 4)
                cv2.line(image, (x_min-pad, y_min-pad), (x_min-pad+len_corner, y_min-pad), box_color, 4)

            if not visible or not within_frame:
                feedback_text = "Body not visible"
                ready = False
            elif too_close:
                feedback_text = "Move Back"
                speak("Move back") 
                ready = False
            elif too_far:
                feedback_text = "Come Closer"
                speak("Come closer") 
                ready = False
            else:
                ready = True

            if ready:
                shoulder_pt = [shoulder.x, shoulder.y]
                elbow_pt = [elbow.x, elbow.y]
                wrist_pt = [wrist.x, wrist.y]
                
                current_angle = calculate_angle(shoulder_pt, elbow_pt, wrist_pt)

                if last_elbow_x is not None:
                    elbow_shift = abs(elbow.x - last_elbow_x) * w
                    bad_form = elbow_shift > 40
                last_elbow_x = elbow.x
                wrist_movement = abs(wrist.x - elbow.x) * w
                bad_form = bad_form or wrist_movement > 120

                cv2.circle(image, tuple(np.multiply(elbow_pt, [w, h]).astype(int)), 10, COLOR_ACCENT, -1)
                cv2.circle(image, tuple(np.multiply(elbow_pt, [w, h]).astype(int)), 15, (255,255,255), 2)

                angle_change = current_angle - prev_angle
                is_extending = angle_change > 3 
                is_curling = angle_change < -3 

                if not bad_form:
                    if current_angle > 160:
                        if stage != "down": speak("Down") 
                        stage = "down"
                        angle_feedback = "" 
                    if current_angle < 30 and stage == "down":
                        if stage != "up": speak(str(counter + 1)) 
                        stage = "up"
                        counter += 1
                        angle_feedback = "" 
                    
                    if stage == "down" and is_extending and current_angle < 140:
                         angle_feedback = "KEEP GOING UP"
                         speak("Keep going up") 

                    if stage == "up" and is_curling and current_angle > 50:
                         angle_feedback = "KEEP GOING DOWN"
                         speak("KEEP GOING DOWN") 

                prev_angle = current_angle

                if bad_form:
                    feedback_text = "FIX ELBOW"
                    speak("Fix your elbow") 
                elif stage == "up":
                    feedback_text = "HOLD / DOWN"
                else:
                    feedback_text = "CURL UP"

            mp_drawing.draw_landmarks(
                image, 
                results.pose_landmarks, 
                mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=COLOR_ACCENT, thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255,255,255), thickness=2, circle_radius=2)
            )

        draw_info_panel(image, counter, stage, feedback_text, angle_feedback)
        
        if ready:
            draw_progress_bar(image, current_angle)

        cv2.imshow(WINDOW_NAME, image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()